In [1]:
!nvidia-smi
!nvcc --version

Sun Jun 21 11:19:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile VectorExample.java

import java.util.Arrays;

public class VectorExample {
    static {
        System.loadLibrary("vectorlib");
    }

    public native int sumVector(int[] arr);

    public static void main(String[] args) {
        int[] data = {1, 2, 3, 4, 5};

        VectorExample example = new VectorExample();
        int result = example.sumVector(data);

        System.out.println("Input vector: " + Arrays.toString(data));
        System.out.println("Sum of vector: " + result);
    }
}

Writing VectorExample.java


In [3]:
%%writefile vectorlib.cu

#include <jni.h>
#include <cuda_runtime.h>

__global__ void sumKernel(const int* data, int length, int* result) {
    int index = blockIdx.x * blockDim.x + threadIdx.x;

    if (index < length) {
        atomicAdd(result, data[index]);
    }
}

extern "C" JNIEXPORT jint JNICALL
Java_VectorExample_sumVector(JNIEnv* env, jobject obj, jintArray arr) {
    int length = env->GetArrayLength(arr);
    jint* input = env->GetIntArrayElements(arr, nullptr);

    int* deviceInput;
    int* deviceResult;
    int result = 0;

    cudaMalloc(&deviceInput, length * sizeof(int));
    cudaMalloc(&deviceResult, sizeof(int));

    cudaMemcpy(
        deviceInput,
        input,
        length * sizeof(int),
        cudaMemcpyHostToDevice
    );

    cudaMemset(deviceResult, 0, sizeof(int));

    int threads = 256;
    int blocks = (length + threads - 1) / threads;

    sumKernel<<<blocks, threads>>>(deviceInput, length, deviceResult);

    cudaDeviceSynchronize();

    cudaMemcpy(
        &result,
        deviceResult,
        sizeof(int),
        cudaMemcpyDeviceToHost
    );

    cudaFree(deviceInput);
    cudaFree(deviceResult);

    env->ReleaseIntArrayElements(arr, input, JNI_ABORT);

    return result;
}

Writing vectorlib.cu


In [4]:
!javac VectorExample.java

!JAVA_HOME=$(dirname $(dirname $(readlink -f $(which javac)))) && \
nvcc -shared -Xcompiler -fPIC \
-I"$JAVA_HOME/include" \
-I"$JAVA_HOME/include/linux" \
vectorlib.cu -o libvectorlib.so

!java -Djava.library.path=. VectorExample

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Input vector: [1, 2, 3, 4, 5]
Sum of vector: 15
